# 01. Seu primeiro experimento

Este notebook percorre um experimento completo com `skxperiments`: **desenhar** a randomização, **estimar** o efeito e **inferir** se ele é real.

> Filosofia da biblioteca: *o mecanismo de atribuição do tratamento é o ponto de partida*. Primeiro decidimos **como** randomizar; o modelo estatístico vem depois.

## 1. Potential outcomes

A base conceitual é o *Rubin Causal Model*: cada unidade tem dois resultados potenciais, `Y(0)` (sem tratamento) e `Y(1)` (com tratamento). Só observamos **um** deles por unidade; o outro é contrafactual.

Para o exemplo, simulamos os dois com um efeito verdadeiro constante de `+0.5`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 200

y0 = rng.normal(size=n)   # Y(0): resultado sem tratamento
tau = 0.5                 # efeito verdadeiro (constante)
y1 = y0 + tau             # Y(1): resultado com tratamento

df = pd.DataFrame({"x": rng.normal(size=n)})   # uma covariavel pre-experimento
df.head()

## 2. Desenho: randomização completa (CRD)

`CRD` atribui metade das unidades ao tratamento, ao acaso. A randomização é o que nos permite atribuir diferenças de resultado ao tratamento, e não a confundidores.

In [ ]:
from skxperiments.design.crd import CRD

design = CRD(p=0.5, seed=42)
assignment = design.randomize(df)
assignment.n_treated_, assignment.n_control_

## 3. O que observamos

Revelamos `Y(1)` para os tratados e `Y(0)` para os controles. É o único resultado que existe na prática. O outcome **pertence aos dados**, então construímos um novo `Assignment` com a coluna `y` (sem mutar o original).

In [ ]:
from skxperiments.core.assignment import CRDAssignment

t = assignment.data_[assignment.treatment_col_].to_numpy()
observed = assignment.data_.copy()
observed["y"] = np.where(t == 1, y1, y0)

assignment = CRDAssignment(
    data=observed,
    treatment_col=assignment.treatment_col_,
    design=design,
    seed=42,
)

## 4. Estimar o ATE

`DifferenceInMeans` é o estimador mais simples: média dos tratados menos média dos controles. Deve ficar perto do efeito verdadeiro `0.5`.

In [ ]:
from skxperiments.estimators.difference_in_means import DifferenceInMeans

result = DifferenceInMeans(outcome_col="y").fit(assignment).estimate()
print(f"ATE estimado: {result.ate:.3f}  (verdadeiro: {tau})")

## 5. Inferência baseada em randomização

O `RandomizationTest` testa a hipótese nula *forte* de Fisher (nenhum efeito em nenhuma unidade) repetindo a randomização milhares de vezes. O p-valor não depende de pressupostos distribucionais; depende só do mecanismo de randomização que **nós** escolhemos.

In [ ]:
from skxperiments.inference import RandomizationTest

rt = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="y"),
    n_permutations=500,
    seed=0,
)
res = rt.fit(assignment).estimate()
print(f"ATE: {res.ate:.3f}  |  p-valor: {res.p_value:.4f}")

## 6. Visualizar a distribuição nula

O ATE observado (linha vermelha) deve cair na cauda da distribuição dos ATEs sob a nula. É isso que torna o efeito "significativo".

In [ ]:
from skxperiments.reporting import plot_null_distribution

ax = plot_null_distribution(res)
ax.figure

## O que aprendemos

- O efeito causal é definido por **potential outcomes**; a randomização revela um deles por unidade.
- `CRD`, `DifferenceInMeans` e `RandomizationTest` formam o fluxo mínimo de ponta a ponta.
- O p-valor de randomização não assume normalidade; vem do próprio desenho.

**Próximo:** `02. Inferência de três jeitos` compara randomização, Neyman (população finita) e bootstrap (superpopulação): quando usar cada um.